# 7b - Alternative Figure 7: 3-predictor model (CIDER + TMB + Pretreatment)

**Purpose**: An alternative to Figure 7, independent of `7_predicted_response_landscape.ipynb`
-- this notebook is fully self-contained
and shares no in-memory objects with either. The underlying MODEL changes
here, not just how a panel is drawn: pretreatment status (prior systemic
therapy, yes/no) is added as a third predictor alongside CIDER T-cell Score
and TMB -- `response ~ cider_log2 + tmb_log2 + pretreated_n` -- fit the same
way as the canonical 2-predictor model in `9_roc.ipynb` (same pooled,
deduplicated ICI-treated training population; same 10-fold x 10-repeat CV
for the reported AUC; `set.seed(42)` for reproducibility), just with one
more term.

All three panels are rebuilt against this new model, each split by
pretreatment status via faceting (not a 4-way dodge -- an earlier attempt to
dodge Responder/Non-Responder x Pretreated/Not Pretreated within each
cohort's x-position collapsed the per-cohort breakdown into 4 boxes with no
visible cohort separation, so faceting is used instead):
- **Panel A'** -- a new 3-input model schematic (`model_schematic_3factor.svg`)
- **Panel B'** -- ICI-treated cohorts, faceted by Pretreated / Not Pretreated,
  each facet showing every cancer-type cohort split by Responder/Non-Responder
  (same 2-group design as the main Panel B, just duplicated per facet)
- **Panel C'** -- the extrapolated landscape on all untreated Hartwig
  patients, faceted by Pretreated / Not Pretreated, using the 3-predictor
  model's predictions (same boxplot style as the main Panel C, no
  external-study overlay)

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cpi_ready.rds`, `CIDER_PROJ_DIR/ready.rds`, `fig_references/schematic_figs/model_schematic_3factor.svg` |
| **Outputs** | `figs/tmb_cider_pretreat_model.rds`, `figs/model_schematic_3factor_panel.rds`, `figs/predicted_response_cpi_cohorts_pretreat.rds`, `figs/predicted_response_landscape_pretreat.rds` / `.png` |
| **Run after** | `4_base.ipynb` (for `ready.rds` / `cpi_ready.rds`) -- does NOT require `9_roc.ipynb` |
| **Then run** | `assemble_nonresponse_figure_pretreat.R` -→ `figure_7_pretreat.png` |

Not part of the main `figure_7.png`.

In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}

source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))

library(patchwork)
library(magick)   # schematic rasterization
library(caret)    # cross-validated AUC

## 0 - Load data

In [ ]:
cpi   <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds"))
ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds"))

## 1 - Fit the 3-predictor model (CIDER + TMB + Pretreatment)

In [ ]:
# Same pooled, deduplicated ICI-treated training population as the canonical
# 2-predictor model (9_roc.ipynb's roc_data_pooled): cohort_broad != "ICI-Treated"
# (drops the pan-cancer duplicate every ICI patient has, see 4_base.ipynb)
# then distinct(sampleId). pretreated_n follows the same == 1 coercion used
# in 0_prepare/5_models.ipynb (pretreated is numeric/character 1/0, not text).
pretreat_base_df <- cpi |>
  dplyr::filter(gp == "T-cell", !is.na(response), cohort_broad != "ICI-Treated") |>
  dplyr::distinct(sampleId, .keep_all = TRUE)

pretreat_train_df <- pretreat_base_df |>
  dplyr::transmute(
    sampleId,
    response     = as.integer(response),
    cider_log2   = log2(cider_score_depth_adj + 1),
    tmb_log2     = log2(tmbPerMb + 1),
    pretreated_n = as.integer(pretreated == 1)
  ) |>
  tidyr::drop_na(response, cider_log2, tmb_log2, pretreated_n)

message("3-predictor (CIDER + TMB + Pretreatment) training N = ", nrow(pretreat_train_df),
        " of ", nrow(pretreat_base_df), " ICI-treated patients with known response (",
        nrow(pretreat_base_df) - nrow(pretreat_train_df), " excluded for missing pretreatment status)")

# Cross-validated AUC (10-fold x 10 repeats) -- identical method/settings to
# 9_roc.ipynb's combined-model CV, just with pretreated_n added to the formula.
set.seed(42)

pretreat_cv_ctrl <- caret::trainControl(
  method          = "repeatedcv",
  number          = 10,
  repeats         = 10,
  classProbs      = TRUE,
  summaryFunction = caret::twoClassSummary,
  savePredictions = "final"
)

pretreat_cv_data <- pretreat_train_df |>
  dplyr::mutate(response_fct = factor(ifelse(response == 1, "Responder", "NonResponder"),
                                      levels = c("Responder", "NonResponder")))

cv_fit_pretreat <- caret::train(
  response_fct ~ cider_log2 + tmb_log2 + pretreated_n,
  data      = pretreat_cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = pretreat_cv_ctrl
)

pretreat_cv_mean <- signif(cv_fit_pretreat$results$ROC,  2)
pretreat_cv_sd   <- round(cv_fit_pretreat$results$ROCSD, 2)
message("CV AUC (10-fold x 10 repeats), CIDER + TMB + Pretreatment: ",
        pretreat_cv_mean, " +/- ", pretreat_cv_sd)

# Final model fit on the full training set (not refit per CV fold) -- this is
# the object used for all three panels' predictions below.
model_pretreat <- glm(response ~ cider_log2 + tmb_log2 + pretreated_n,
                      data = pretreat_train_df, family = binomial())
pretreat_cv_auc_label <- paste0(pretreat_cv_mean, " ± ", pretreat_cv_sd)

saveRDS(
  list(model = model_pretreat, data = pretreat_train_df,
       cv_auc_mean = pretreat_cv_mean, cv_auc_sd = pretreat_cv_sd,
       n_train = nrow(pretreat_train_df)),
  paste0(CIDER_PROJ_DIR, "figs/tmb_cider_pretreat_model.rds")
)

## 2 - Panel A': 3-input model schematic

In [ ]:
# Same rasterization approach as 7_predicted_response_landscape.ipynb's main
# Panel A (image_read_svg(), no `density` argument -- that belongs only to
# the generic magick::image_read()), pointed at the 3-input schematic asset.
schematic_3f_path <- file.path(dirname(getwd()), "fig_references", "schematic_figs", "model_schematic_3factor.svg")
if (!file.exists(schematic_3f_path))
  stop("Missing schematic: fig_references/schematic_figs/model_schematic_3factor.svg not found in repo root.")
schematic_3f_img <- magick::image_read_svg(schematic_3f_path, width = 6000)
message("3-factor schematic rasterized at: ",
        paste(magick::image_info(schematic_3f_img)[c("width", "height")], collapse = "x"), "px")

p_schematic_3f <- ggplot2::ggplot() +
  ggplot2::annotation_raster(as.raster(schematic_3f_img), xmin = 0, xmax = 1, ymin = 0, ymax = 1) +
  ggplot2::scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
  ggplot2::scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  ggplot2::theme_void()

saveRDS(p_schematic_3f, paste0(CIDER_PROJ_DIR, "figs/model_schematic_3factor_panel.rds"))
set_plot_dims(10, 4.6); p_schematic_3f

## 3 - Panel B': ICI-treated cohorts, faceted by pretreatment status

Faceted into "Pretreated" / "Not Pretreated" panels -- each facet is the same
2-group (Responder/Non-Responder) dodge design as the main Panel B, just
restricted to that pretreatment stratum. (An earlier version tried to dodge
all 4 Response x Pretreatment combinations within each cohort's x-position
using fill + alpha; it collapsed to 4 boxes with no visible per-cohort
separation, so faceting is used instead.)

In [ ]:
# ── Panel B': CPI cohorts, predicted probability, faceted by pretreatment ────
panel_b3_df <- cpi |>
  dplyr::filter(gp == "T-cell", cohort_broad %in% BROAD_LEVELS) |>
  dplyr::transmute(
    sampleId,
    cohort_broad,
    cohort_broad_go,
    response_int = as.integer(response),
    cider_log2   = log2(cider_score_depth_adj + 1),
    tmb_log2     = log2(tmbPerMb + 1),
    pretreated_n = as.integer(pretreated == 1)
  ) |>
  tidyr::drop_na(response_int, cider_log2, tmb_log2, pretreated_n)

panel_b3_df$pred_prob <- predict(model_pretreat, newdata = panel_b3_df, type = "response")

panel_b3_df <- panel_b3_df |>
  dplyr::mutate(
    response_label   = factor(ifelse(response_int == 1, "Responder", "Non-Responder"),
                              levels = c("Non-Responder", "Responder")),
    pretreated_label  = factor(ifelse(pretreated_n == 1, "Pretreated", "Not Pretreated"),
                               levels = c("Not Pretreated", "Pretreated")),
    cohort_broad_go   = factor(cohort_broad_go, levels = levels(cpi$cohort_broad_go))
  )

n_panel_b3 <- panel_b3_df |>
  dplyr::filter(cohort_broad == "ICI-Treated") |>
  dplyr::distinct(sampleId) |>
  nrow()

med_labels_b3 <- panel_b3_df |>
  dplyr::group_by(cohort_broad_go, pretreated_label, response_label) |>
  dplyr::summarise(med = median(pred_prob, na.rm = TRUE), .groups = "drop") |>
  dplyr::mutate(med_label = sprintf("%.0f%%", med * 100))

count_labels_b3 <- panel_b3_df |>
  dplyr::group_by(cohort_broad_go, pretreated_label, response_label) |>
  dplyr::summarise(n = dplyr::n(), .groups = "drop") |>
  dplyr::mutate(n_label = paste0("n = ", n))

p_panel_b3 <- panel_b3_df |>
  ggplot2::ggplot(ggplot2::aes(x = cohort_broad_go, y = pred_prob, fill = response_label)) +
  # Same Responder/Non-Responder palette as 7_predicted_response_landscape.ipynb
  # (helpers/themes.r's responder_colors), not hardcoded colors[5].
  ggplot2::annotate("rect", xmin = -Inf, xmax = Inf, ymin = 0, ymax = 0.05,
                    fill = unname(responder_colors["Non-Responder"]), alpha = 0.08) +
  ggplot2::geom_hline(yintercept = 0.5, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.5) +
  ggplot2::geom_jitter(
    data = panel_b3_df |>
      dplyr::group_by(cohort_broad_go, pretreated_label, response_label) |>
      dplyr::slice_sample(n = 100) |>
      dplyr::ungroup(),
    ggplot2::aes(colour = response_label),
    position = ggplot2::position_jitterdodge(jitter.width = 0.12, dodge.width = 0.8),
    alpha = 0.2, size = 0.35, show.legend = FALSE
  ) +
  ggplot2::scale_colour_manual(
    values = responder_colors, guide = "none"
  ) +
  ggplot2::geom_boxplot(outlier.shape = NA, alpha = 0.7,
                        position = ggplot2::position_dodge(width = 0.8)) +
  ggplot2::geom_text(
    data = med_labels_b3,
    ggplot2::aes(x = cohort_broad_go, y = med, label = med_label, group = response_label),
    inherit.aes = FALSE, position = ggplot2::position_dodge(width = 0.8),
    vjust = -0.5, size = 2.0, fontface = "bold", colour = COL_INK
  ) +
  ggplot2::geom_text(
    data = count_labels_b3,
    ggplot2::aes(x = cohort_broad_go, y = 0.015, label = n_label, group = response_label),
    inherit.aes = FALSE, position = ggplot2::position_dodge(width = 0.8),
    vjust = 0, size = 1.9, colour = COL_INK
  ) +
  ggplot2::scale_fill_manual(
    values = responder_colors, name = NULL
  ) +
  ggplot2::scale_y_continuous(
    limits = c(0, NA), breaks = c(0.05, seq(0, 1, by = 0.2)),
    labels = scales::percent_format(accuracy = 1),
    expand = ggplot2::expansion(mult = c(0, 0.12))
  ) +
  ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 2)) +
  ggplot2::facet_wrap(~ pretreated_label, ncol = 2) +
  go_theme +
  ggplot2::theme(
    axis.text.x     = ggplot2::element_text(size = 7),
    plot.subtitle   = ggplot2::element_text(hjust = 0.5),
    legend.position = "right",
    strip.text      = ggplot2::element_text(face = "bold")
  ) +
  ggplot2::guides(fill = ggplot2::guide_legend(title = NULL)) +
  ggplot2::labs(
    title    = "Predicted Response Probability — ICI-Treated Cohorts",
    subtitle = paste0("CPI patients with known response and pretreatment status, n = ",
                      format(n_panel_b3, big.mark = ","),
                      "; model: CIDER + TMB + Pretreatment (CV AUC ", pretreat_cv_auc_label, ")"),
    x        = NULL,
    y        = "Predicted Response Probability"
  ) +
  tags_supp

set_plot_dims(12, 5.5); p_panel_b3
saveRDS(p_panel_b3, paste0(CIDER_PROJ_DIR, "figs/predicted_response_cpi_cohorts_pretreat.rds"))

## 4 - Panel C': extrapolated landscape, faceted by pretreatment status

Same boxplot design as the main Panel C (shaded low/high reference bands,
background jitter, median labels, no external-study overlay), faceted into
"Pretreated" / "Not Pretreated" panels. Cohorts are ordered once from the
pooled (both-pretreatment) median so the x-axis order matches between the
two facets.

In [ ]:
# ── Panel C': untreated Hartwig patients, 3-predictor model, faceted ─────────
predict3_base_df <- ready |>
  dplyr::filter(gp == "T-cell", cohort_pan %in% PAN_LEVELS) |>
  dplyr::distinct(sampleId, .keep_all = TRUE)

predict3_df <- predict3_base_df |>
  dplyr::transmute(
    sampleId, cohort_pan, cpi,
    cider_log2   = log2(cider_score_depth_adj + 1),
    tmb_log2     = log2(tmbPerMb + 1),
    pretreated_n = as.integer(pretreated == 1)
  ) |>
  tidyr::drop_na(cider_log2, tmb_log2, pretreated_n)

message("Panel C' (3-predictor) patients with predictions: ", nrow(predict3_df),
        " of ", nrow(predict3_base_df), " T-cell-group patients (",
        nrow(predict3_base_df) - nrow(predict3_df), " excluded for missing predictor, incl. pretreatment status)")

predict3_df$pred_prob <- predict(model_pretreat, newdata = predict3_df, type = "response")

panel_c3_df <- predict3_df |>
  dplyr::filter(!cpi, cohort_pan %in% PAN_LEVELS) |>
  dplyr::mutate(
    pretreated_label = factor(ifelse(pretreated_n == 1, "Pretreated", "Not Pretreated"),
                              levels = c("Not Pretreated", "Pretreated"))
  ) |>
  dplyr::select(sampleId, cohort_pan, pretreated_label, pred_prob)

# Cohort order from the POOLED median (both pretreatment strata together) so
# the x-axis lines up between the two facets.
cohort_summary_c3 <- panel_c3_df |>
  dplyr::group_by(cohort_pan) |>
  dplyr::summarise(n = dplyr::n(), med = median(pred_prob, na.rm = TRUE), .groups = "drop") |>
  dplyr::arrange(dplyr::desc(med)) |>
  dplyr::mutate(label = paste0(cohort_pan, "\n(N = ", format(n, big.mark = ",", trim = TRUE), ")"))

panel_c3_df <- panel_c3_df |>
  dplyr::left_join(dplyr::select(cohort_summary_c3, cohort_pan, label), by = "cohort_pan") |>
  dplyr::mutate(label = factor(label, levels = cohort_summary_c3$label))

med_labels_c3 <- panel_c3_df |>
  dplyr::group_by(label, pretreated_label) |>
  dplyr::summarise(med = median(pred_prob, na.rm = TRUE), .groups = "drop") |>
  dplyr::mutate(med_label = sprintf("%.0f%%", med * 100))

p_panel_c3 <- panel_c3_df |>
  ggplot2::ggplot(ggplot2::aes(x = label, y = pred_prob)) +
  # Same Responder (orange) / Non-Responder (grey) reference-band palette
  # as 7_predicted_response_landscape.ipynb's Panel C, not hardcoded
  # colors[5]/[6].
  ggplot2::annotate("rect", xmin = -Inf, xmax = Inf, ymin = 0, ymax = 0.05,
                    fill = unname(responder_colors["Non-Responder"]), alpha = 0.08) +
  ggplot2::annotate("rect", xmin = -Inf, xmax = Inf, ymin = 0.5, ymax = 1,
                    fill = unname(responder_colors["Responder"]), alpha = 0.05) +
  ggplot2::geom_hline(yintercept = 0.5, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.5) +
  ggplot2::geom_jitter(
    data = panel_c3_df |> dplyr::group_by(label, pretreated_label) |> dplyr::slice_sample(n = 100) |> dplyr::ungroup(),
    width = 0.18, alpha = 0.12, size = 0.3, colour = COL_NEGATIVE
  ) +
  ggplot2::geom_boxplot(fill = COL_NEGATIVE, colour = COL_INK, outlier.shape = NA, alpha = 0.7) +
  ggplot2::geom_text(
    data = med_labels_c3, ggplot2::aes(x = label, y = med, label = med_label),
    inherit.aes = FALSE, vjust = -0.5, size = 2.0, fontface = "bold", colour = COL_INK
  ) +
  ggplot2::scale_y_continuous(
    limits = c(0, 1), breaks = c(0, 0.05, 0.5, 1),
    labels = scales::percent_format(accuracy = 1),
    expand = ggplot2::expansion(mult = c(0.01, 0.03))
  ) +
  ggplot2::facet_wrap(~ pretreated_label, ncol = 2) +
  go_theme +
  ggplot2::theme(
    axis.text.x   = ggplot2::element_text(angle = 45, hjust = 1, size = 7),
    plot.subtitle = ggplot2::element_text(hjust = 0.5),
    strip.text    = ggplot2::element_text(face = "bold")
  ) +
  ggplot2::labs(
    title    = paste0("Extrapolated ICI Response Probability | Model AUC (",
                      pretreat_cv_auc_label, ") with ICI Treated Patients"),
    subtitle = paste0("All Hartwig patients without ICI treatment, n = ",
                      format(nrow(panel_c3_df), big.mark = ","),
                      "; model: CIDER + TMB + Pretreatment"),
    x        = "Cohort",
    y        = "Extrapolated Response Probability"
  ) +
  tags_supp

set_plot_dims(16, 6.5); p_panel_c3
saveRDS(p_panel_c3, paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape_pretreat.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape_pretreat.png"),
       p_panel_c3, width = 16, height = 6.5, dpi = 300)
message("Saved predicted_response_landscape_pretreat.png/.rds (feeds figure_7_pretreat.png)")